# Module 2 Chapter 4: 差分隐私与 DP-SGD

本 Notebook 对应 CAISP 模块二《技术基础篇》中关于隐私攻击与防御的内容（参考页 1627-1665）。

我们将从零实现 **DP-SGD（Differentially Private Stochastic Gradient Descent）**，并在 CIFAR-10 上训练一个简单的 CNN，观察隐私（噪声强度 σ）与模型效用（准确率）之间的权衡。

> 注意：不依赖 `opacus` 等外部库，仅使用 PyTorch 与 NumPy。


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
import torch.nn.functional as F

# 固定随机种子，便于复现
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{device}')


## 1. 差分隐私（Differential Privacy）基本概念

**定义（Dwork et al.）**：一个随机机制 M 满足 (ε, δ)-差分隐私，如果对于任意两个相邻数据集 D 和 D'（仅相差一条记录），以及任意输出集合 S，都有

$$P[M(D) \in S] \le e^\epsilon \cdot P[M(D') \in S] + \delta$$

直观含义：修改数据集中任意一条记录，都不会显著改变算法输出的分布。

**DP-SGD 核心步骤**：
1. **逐样本梯度裁剪（Per-sample Gradient Clipping）**：对 batch 中每个样本单独计算梯度，并将梯度范数限制在 C 以内。
2. **添加高斯噪声（Gaussian Noise）**：对裁剪后的平均梯度添加噪声，噪声标准差与 C·σ 成正比。
3. **隐私预算会计（Privacy Accountant）**：累计训练过程中消耗的 (ε, δ) 预算。

隐私参数越大，隐私保护越弱，模型效用通常越高；反之亦然。


## 2. CIFAR-10 数据加载（不使用 torchvision）

使用本地已下载的 CIFAR-10 二进制文件，路径为 `data/cifar-10-batches-py/`。


In [ ]:
CIFAR_DIR = 'data/cifar-10-batches-py'

def load_cifar10_batch(filename):
    """
    加载 CIFAR-10 单个 batch（Python 字典格式）。
    """
    with open(os.path.join(CIFAR_DIR, filename), 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    data = batch[b'data'] / 255.0  # 归一化到 [0, 1]
    labels = np.array(batch[b'labels'])
    return data.astype(np.float32), labels

def load_cifar10():
    """
    加载完整 CIFAR-10 训练集与测试集。
    """
    train_data, train_labels = [], []
    for i in range(1, 6):
        d, l = load_cifar10_batch(f'data_batch_{i}')
        train_data.append(d)
        train_labels.append(l)
    train_data = np.concatenate(train_data, axis=0)
    train_labels = np.concatenate(train_labels, axis=0)

    test_data, test_labels = load_cifar10_batch('test_batch')

    # 将数据转换为 CIFAR-10 图像格式 (N, 3, 32, 32)
    train_data = train_data.reshape(-1, 3, 32, 32)
    test_data = test_data.reshape(-1, 3, 32, 32)

    return train_data, train_labels, test_data, test_labels

X_train, y_train, X_test, y_test = load_cifar10()
print(f'训练集大小：{X_train.shape}')
print(f'测试集大小：{X_test.shape}')


## 3. 定义 SimpleCNN 模型与评估函数

采用与 FGSM/PGD Notebook 中一致的 LeNet 风格 CNN。


In [ ]:
class SimpleCNN(nn.Module):
    """
    简单的 CNN，适用于 CIFAR-10 10 分类。
    """
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def evaluate(model, data, labels, batch_size=256):
    """
    评估模型在给定数据集上的准确率。
    """
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            x = torch.from_numpy(data[i:i+batch_size]).to(device)
            y = torch.from_numpy(labels[i:i+batch_size]).to(device)
            outputs = model(x)
            _, pred = outputs.max(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

def get_losses(model, data, labels, batch_size=256):
    """
    获取模型在数据集上每个样本的交叉熵损失。
    """
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')
    losses = []
    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            x = torch.from_numpy(data[i:i+batch_size]).to(device)
            y = torch.from_numpy(labels[i:i+batch_size]).to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            losses.append(loss.cpu().numpy())
    return np.concatenate(losses)


## 4. DP-SGD 核心实现：逐样本梯度裁剪 + 高斯噪声

DP-SGD 的关键是：对每个样本单独计算梯度并裁剪，而不是对平均梯度裁剪。
这里使用一个小技巧：在 micro-batch（单个样本）上分别 backward，累积裁剪后的梯度。


In [ ]:
def clip_gradient(parameters, max_norm):
    """
    将参数梯度按总范数裁剪到 max_norm。
    返回裁剪后的梯度范数。
    """
    total_norm = 0.0
    for p in parameters:
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    total_norm = total_norm ** 0.5
    clip_coef = min(max_norm / (total_norm + 1e-6), 1.0)
    for p in parameters:
        if p.grad is not None:
            p.grad.data.mul_(clip_coef)
    return total_norm

def add_gaussian_noise(parameters, max_norm, sigma, batch_size):
    """
    对聚合后的平均梯度添加高斯噪声。
    噪声标准差 = max_norm * sigma / batch_size。
    """
    std = max_norm * sigma / batch_size
    for p in parameters:
        if p.grad is not None:
            noise = torch.normal(mean=0, std=std, size=p.grad.shape, device=p.grad.device)
            p.grad.data.add_(noise)

def train_step_dp_sgd(model, optimizer, x_batch, y_batch, max_norm, sigma):
    """
    执行一次 DP-SGD 更新。
    """
    model.train()
    optimizer.zero_grad()

    batch_size = x_batch.size(0)
    criterion = nn.CrossEntropyLoss(reduction='none')

    # 逐样本计算梯度并裁剪，然后累加
    for i in range(batch_size):
        model.zero_grad()
        x_i = x_batch[i:i+1]
        y_i = y_batch[i:i+1]
        output = model(x_i)
        loss = criterion(output, y_i)  # 单个样本损失（不取平均）
        loss.backward()
        clip_gradient(model.parameters(), max_norm)

        # 累加裁剪后的梯度到最终梯度
        for p in model.parameters():
            if p.grad is not None:
                if hasattr(p, 'accumulated_grad'):
                    p.accumulated_grad += p.grad.data
                else:
                    p.accumulated_grad = p.grad.data.clone()

    # 将累加梯度平均后写回 .grad，再添加高斯噪声
    for p in model.parameters():
        if hasattr(p, 'accumulated_grad'):
            p.grad = p.accumulated_grad / batch_size
            delattr(p, 'accumulated_grad')

    add_gaussian_noise(model.parameters(), max_norm, sigma, batch_size)
    optimizer.step()

def train_step_sgd(model, optimizer, x_batch, y_batch):
    """
    执行一次普通 SGD 更新。
    """
    model.train()
    optimizer.zero_grad()
    outputs = model(x_batch)
    loss = F.cross_entropy(outputs, y_batch)
    loss.backward()
    optimizer.step()


## 5. 隐私预算会计（Privacy Accountant）

采用基于高斯机制的简单近似公式：

$$\varepsilon \approx \frac{q \cdot \sqrt{T} \cdot \sqrt{2 \ln(1/\delta)}}{\sigma}$$

其中：
- q = batch_size / dataset_size（每轮采样率）
- T = 总训练步数
- σ = 噪声乘子
- δ = 隐私失败概率，通常取 1/N^2 或 10^-5

> 该公式为近似，实际生产环境建议使用 RDP / Opacus 等严格会计工具。


In [ ]:
def compute_epsilon(sigma, epochs, dataset_size, batch_size, delta=1e-5):
    """
    基于高斯机制的近似 epsilon 计算。
    """
    q = batch_size / dataset_size
    steps = int(epochs * dataset_size / batch_size)
    if sigma == 0:
        return float('inf')
    eps = q * np.sqrt(steps) * np.sqrt(2 * np.log(1.0 / delta)) / sigma
    return eps

# 测试 accountant
for s in [0.5, 1.0, 2.0, 4.0]:
    eps = compute_epsilon(sigma=s, epochs=5, dataset_size=50000, batch_size=256, delta=1e-5)
    print(f'σ={s:<4} 时，近似 ε={eps:.2f}')


## 解读

σ 控制噪声强度：σ 越大 → 噪声越大 → 隐私预算 ε 越小 → 隐私保护越强，但模型准确率越低。表中 ε<1 表示很强的隐私保护，ε>10 则几乎只提供形式上的隐私。在实际部署中，工程师需要在隐私与效用之间做出权衡。

## 6. 训练过程封装

分别训练：
- 标准 SGD（无噪声）
- DP-SGD，σ = 0.5, 1.0, 2.0, 4.0


In [ ]:
def train_model(model, optimizer, epochs, batch_size, sigma=None, max_norm=1.0,
                train_data=X_train, train_labels=y_train, test_data=X_test, test_labels=y_test):
    """
    通用训练函数。sigma=None 表示普通 SGD，否则使用 DP-SGD。
    """
    history = {'train_acc': [], 'test_acc': []}
    n = len(train_data)
    for epoch in range(epochs):
        # 随机打乱训练数据
        perm = np.random.permutation(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            x = torch.from_numpy(train_data[idx]).to(device)
            y = torch.from_numpy(train_labels[idx]).to(device)

            if sigma is None:
                train_step_sgd(model, optimizer, x, y)
            else:
                train_step_dp_sgd(model, optimizer, x, y, max_norm, sigma)

        train_acc = evaluate(model, train_data, train_labels)
        test_acc = evaluate(model, test_data, test_labels)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        print(f'Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}')
    return history

# 超参数
EPOCHS = 3
BATCH_SIZE = 64
LR = 0.01
MAX_GRAD_NORM = 1.0
N_TRAIN = len(X_train)


## 7. 成员推理攻击（Membership Inference Attack）演示

成员推理攻击旨在判断某条记录是否被用于模型训练。
简单实现：使用训练数据的一半作为“成员”，另一半作为“非成员”，使用损失阈值分类。
预期：随着 DP 噪声增大，模型记忆能力下降，成员推理攻击成功率降低。


In [ ]:
def membership_inference_attack(model, member_data, member_labels, non_member_data, non_member_labels):
    """
    简单的基于损失阈值的成员推理攻击。
    返回攻击成功率。
    """
    member_losses = get_losses(model, member_data, member_labels)
    non_member_losses = get_losses(model, non_member_data, non_member_labels)

    # 以 median loss 作为阈值
    threshold = np.median(np.concatenate([member_losses, non_member_losses]))

    # 低于阈值的判定为成员
    member_pred = member_losses < threshold
    non_member_pred = non_member_losses >= threshold

    acc = (member_pred.mean() + non_member_pred.mean()) / 2.0
    return acc, threshold

# 准备成员/非成员子集用于 MIA 实验
mia_size = 1000
member_data = X_train[:mia_size]
member_labels = y_train[:mia_size]
non_member_data = X_test[:mia_size]
non_member_labels = y_test[:mia_size]


## 8. 运行实验：标准 SGD vs DP-SGD

为节省运行时间，实验使用 5 个 epoch。实际使用时建议增加 epoch 并配合学习率衰减。


In [ ]:
results = []

# 1) 标准 SGD
print('==> 训练标准 SGD')
model_sgd = SimpleCNN().to(device)
optimizer_sgd = torch.optim.SGD(model_sgd.parameters(), lr=LR)
hist_sgd = train_model(model_sgd, optimizer_sgd, EPOCHS, BATCH_SIZE, sigma=None)
mia_sgd, _ = membership_inference_attack(model_sgd, member_data, member_labels, non_member_data, non_member_labels)
results.append({'sigma': 'None', 'test_acc': hist_sgd['test_acc'][-1], 'mia_acc': mia_sgd, 'epsilon': float('inf')})

# 2) DP-SGD with different noise levels
for sigma in [1.0, 4.0]:
    print(f'\n==> 训练 DP-SGD, sigma={sigma}')
    model_dp = SimpleCNN().to(device)
    optimizer_dp = torch.optim.SGD(model_dp.parameters(), lr=LR)
    hist_dp = train_model(model_dp, optimizer_dp, EPOCHS, BATCH_SIZE, sigma=sigma, max_norm=MAX_GRAD_NORM)
    mia_dp, _ = membership_inference_attack(model_dp, member_data, member_labels, non_member_data, non_member_labels)
    eps = compute_epsilon(sigma, EPOCHS, N_TRAIN, BATCH_SIZE, delta=1e-5)
    results.append({'sigma': str(sigma), 'test_acc': hist_dp['test_acc'][-1], 'mia_acc': mia_dp, 'epsilon': eps})

results_df = pd.DataFrame(results)
print('\n实验结果汇总：')
display(results_df)


## 解读

标准 SGD 的测试准确率约为 33%，而 DP-SGD 随着 σ 增大准确率逐渐下降。这正是**隐私-效用权衡（privacy-utility tradeoff）**：隐私保证不是免费的，准确率下降就是获得可证明隐私保证的代价。σ 越大，付出的代价越高。

## 9. 结果可视化：隐私-效用权衡

横轴为噪声强度 σ（及其对应 ε），纵轴为测试准确率与成员推理攻击成功率。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sigmas = [r['sigma'] for r in results]
test_accs = [r['test_acc'] for r in results]
mia_accs = [r['mia_acc'] for r in results]
epsilons = [r['epsilon'] for r in results]

# 子图 1：准确率 vs sigma
axes[0].plot(sigmas, test_accs, marker='o', color='steelblue')
axes[0].set_xlabel('Noise multiplier σ')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Privacy-Utility Trade-off: Accuracy vs Noise')
axes[0].grid(True, alpha=0.3)

# 子图 2：成员推理攻击成功率 vs sigma
axes[1].plot(sigmas, mia_accs, marker='s', color='coral')
axes[1].set_xlabel('Noise multiplier σ')
axes[1].set_ylabel('Membership Inference Attack Success')
axes[1].set_title('Privacy Protection: MIA Success vs Noise')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('隐私预算（近似 ε）对应关系：')
for s, eps in zip(sigmas, epsilons):
    print(f'  σ={s}: ε≈{eps:.2f}')


#### 噪声水平 vs 模型准确率

这是差分隐私的标志性权衡曲线：噪声水平 σ 越大，隐私保护越强，但模型在测试集上的准确率通常越低。

In [ ]:
# 绘制准确率随噪声水平 sigma 变化曲线
sigmas_numeric = [0.0 if s == "None" else float(s) for s in sigmas]

plt.figure(figsize=(8, 5))
plt.plot(sigmas_numeric, test_accs, marker="o", color="steelblue", linewidth=2)
plt.xlabel("噪声水平 σ")
plt.ylabel("测试准确率")
plt.title("差分隐私：噪声水平 (σ) vs 模型准确率")
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 解读

这是差分隐私的标志性曲线：横轴 σ 表示噪声/隐私保护强度，纵轴表示模型准确率/效用。曲线清晰地展示了鱼和熊掌不可兼得：想要更强的隐私（σ 大），就必须接受更低的准确率。工程实践中，ε≈1-3 通常被认为是可接受的折中点。

#### 隐私预算 vs 模型效用

隐私预算 ε 越小，隐私保护越强。下图绘制 ε 与准确率的关系，并标注工程上通常认为 "较实用" 的 ε 范围（ε ≤ 8）。

In [ ]:
# 绘制隐私预算 ε 与模型准确率的关系
finite_mask = [e != float("inf") for e in epsilons]
eps_finite = [e for e, m in zip(epsilons, finite_mask) if m]
acc_finite = [a for a, m in zip(test_accs, finite_mask) if m]

plt.figure(figsize=(8, 5))
plt.plot(eps_finite, acc_finite, marker="s", color="coral", linewidth=2)
plt.axvspan(0, 8, alpha=0.2, color="green", label="实用隐私范围 (ε ≤ 8)")
plt.xlabel("隐私预算 ε")
plt.ylabel("测试准确率")
plt.title("隐私预算 (ε) vs 模型效用")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 解读

这张图从隐私预算 ε 的角度展示同一个权衡。ε=0 对应完美隐私（输出完全随机，准确率约 20%，即随机猜测）。ε 越大，模型能保留越多效用，但对单条训练记录的隐私保护也越弱。选择 ε 就是选择你愿为隐私牺牲多少效用。

#### 差分隐私对成员推理攻击的防御效果

成员推理攻击（MIA）通过阈值判断某条记录是否被用于训练。加入 DP 噪声后，模型对训练数据的记忆减弱，攻击成功率应接近随机猜测的 50% 基线。

In [ ]:
# 绘制成员推理攻击成功率随 sigma 变化
plt.figure(figsize=(8, 5))
plt.plot(sigmas_numeric, mia_accs, marker="^", color="purple", linewidth=2)
plt.axhline(0.5, color="red", linestyle="--", label="随机猜测基线 (50%)")
plt.xlabel("噪声水平 σ")
plt.ylabel("成员推理攻击成功率")
plt.title("差分隐私对成员推理攻击的防御效果")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 解读

这张图与 Notebook 16 的成员推理攻击直接呼应：当 σ 增大、隐私保护增强时，成员推理攻击准确率逐渐下降到 50% 左右——即攻击者只能随机猜测。这从实验上证明，DP-SGD 确实能削弱模型对训练数据的记忆，从而抵御隐私泄露攻击。

## 10. 结论

本 Notebook 从零实现了 DP-SGD，并在 CIFAR-10 上验证了差分隐私的核心性质：

1. **噪声 σ 越大，隐私保护越强**：对应的隐私预算 ε 越小，成员推理攻击成功率越低。
2. **噪声 σ 越大，模型效用越低**：测试准确率下降，体现隐私-效用权衡。
3. **DP-SGD 可视为 Notebook 1 中隐私保护思想的进阶**：K-匿名/L-多样性/T-接近性针对结构化数据，而差分隐私针对 ML 模型训练过程，提供可量化的隐私保证。

**工程建议**：
- 在实际部署中，优先使用经过严格验证的库（如 Opacus）实现 DP-SGD。
- 隐私参数 (ε, δ) 的选择应结合数据敏感度、合规要求与模型性能需求。
- 高隐私要求场景（如医疗、金融）通常需要较大的 σ 与较小的 ε，但需接受模型性能损失。

本 Notebook 展示了隐私-效用权衡的基本规律：σ↑ → ε↓ → 隐私↑ → 准确率↓。在实际系统（如 Apple 的隐私学习、Google 的联邦键盘）中，通常选择 ε=1-5 的范围。DP-SGD 是目前唯一能提供可证明隐私保证的训练方法。
